In [ ]:
from matplotrender import (
    plot_mesh_image,
    plot_image_array_diff3,
    plot_mesh_gouraud,
    plot_mesh_video,
    render_video_mesh_diff,
    render_video_mesh_diffs,
    perspective,
)
import trimesh
import numpy as np

In [ ]:
mesh = trimesh.load('./src/stanford_bunny.obj', force='mesh')
V = np.array(mesh.vertices)
F = np.array(mesh.faces)
print(f'V: {V.shape}, F: {F.shape}')

## plot_mesh_image
Renders a static image of one or more meshes side by side.  
`mode`: `'mesh'` (wireframe) | `'shade'` | `'normal'`

In [ ]:
SIZE = 3
v_list = [V]
f_list = [F]
rot_list = [[0, 30, 0]]

plot_mesh_image(v_list, f_list, rot_list=rot_list, size=SIZE, norm=True, mode='mesh')
plot_mesh_image(v_list, f_list, rot_list=rot_list, size=SIZE, norm=True, mode='normal')
plot_mesh_image(v_list, f_list, rot_list=rot_list, size=SIZE, norm=True, mode='shade')

# custom perspective
plot_mesh_image(v_list, f_list, rot_list=rot_list, custom_perspective=perspective(15, 1, 1, 10), size=SIZE, norm=True)
plot_mesh_image(v_list, f_list, rot_list=rot_list, custom_perspective=perspective(105, 1, 1, 10), size=SIZE, norm=True)

## plot_mesh_gouraud
Gouraud shading via `matplotlib.tri`. Supports numpy and torch tensor inputs.  
Pass `Cs` to visualize per-vertex labels (e.g. segmentation logits).

In [ ]:
# --- numpy inputs ---
plot_mesh_gouraud(
    [V], [F],
    rot_list=[[0, 30, 0]],
    norm=True, mode='shade', bg_black=False, show=True,
)
plot_mesh_gouraud(
    [V], [F],
    rot_list=[[0, 30, 0]],
    norm=True, mode='normal', bg_black=False, show=True,
)

In [ ]:
# --- torch tensor inputs ---
import torch
V_torch = torch.tensor(V, dtype=torch.float32)
F_torch = torch.tensor(F, dtype=torch.long)

plot_mesh_gouraud(
    [V_torch], [F_torch],
    rot_list=[[0, 30, 0]],
    norm=True, mode='shade', bg_black=False, show=True,
)

In [ ]:
# --- segmentation labels as Cs (torch) ---
# Cs: per-vertex logits (V, num_classes) → argmax determines color
num_verts = V.shape[0]
C_seg = torch.randn(num_verts, 5)  # 5 classes

plot_mesh_gouraud(
    [V_torch], [F_torch],
    Cs=[C_seg],
    rot_list=[[0, 30, 0]],
    norm=True, mode='shade', bg_black=False, show=True,
)

## plot_image_array_diff3
Renders multiple meshes with per-face L2 difference from a reference `D` colored on each mesh.

In [ ]:
V_noisy = V + np.random.randn(*V.shape) * 0.001

plot_image_array_diff3(
    [V_noisy], [F],
    D=V,          # reference mesh
    rot=(0, 30, 0),
    norm=True,
    bg_black=False,
)

## plot_mesh_video
Renders a mesh animation as an `.mp4`.  
`Vs`: list of `(T, V, 3)` sequences — each rendered as a separate panel.

In [ ]:
T = 36  # 10 deg/frame → full 360
rot_list = [[0, y, 0] for y in np.linspace(0, 360, T, endpoint=False)]
V_seq = np.tile(V[np.newaxis], (T, 1, 1))  # (T, V, 3)

plot_mesh_video(
    [V_seq], [F],
    rot_list=rot_list,
    norm=True, size=4, fps=12,
    bg_black=False,
    savedir='out', savename='bunny_rotate',
)

## render_video_mesh_diff
Renders a video comparing a reference sequence `D` (left panel) against one or more noisy sequences `Vs` (right panels), colored by per-face L2 error.  
When `D=None`, the first frame of `Vs[0]` is used as a static reference.

In [ ]:
V_noisy_seq = V_seq + np.random.randn(*V_seq.shape) * 0.002

# With explicit GT reference
render_video_mesh_diff(
    [V_noisy_seq], [F],
    D=V_seq,          # GT sequence: left panel
    rot_list=rot_list,
    norm=True, size=4, fps=12,
    bg_black=False,
    savedir='out', savename='bunny_diff_rotate',
)

In [ ]:
# D=None → uses first frame of Vs[0] as static reference
render_video_mesh_diff(
    [V_noisy_seq], [F],
    D=None,
    rot_list=rot_list,
    norm=True, size=4, fps=12,
    bg_black=False,
    savedir='out', savename='bunny_diff_static_ref',
)

## render_video_mesh_diffs
Compares multiple `src` sequences against a single `anchor` sequence frame by frame.  
Layout: `[ anchor | src_0 | src_1 | ... ]`  
Colormap is normalized **globally** across all frames for consistent comparison.

In [ ]:
V_noisy1 = V_seq + np.random.randn(*V_seq.shape) * 0.003
V_noisy2 = V_seq + np.random.randn(*V_seq.shape) * 0.006

render_video_mesh_diffs(
    src_vs=[V_noisy1, V_noisy2],
    src_fs=[F, F],
    anc_vs=V_seq,   # anchor (GT)
    anc_fs=F,
    rot_list=rot_list,
    norm=True, size=4, fps=12,
    bg_black=False,
    savedir='out', savename='bunny_diffs',
)